# RWB Classic ML v8

In [ ]:
%pip install -q numpy==2.5.4 pandas==2.2.3 scipy==1.18.1 lightgbm==4.7.0 pyarrow==26.0.1

In [ ]:
from base64 import b64decode
from io import BytesIO
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import gc
import json
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"

import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.linalg import solve
from scipy.signal import lfilter

PROJECT_DIR = None
RETRAIN = False
RECHECK = False
THREADS = 4
BASE_THREADS = 3

DAY = 48
H = 8
CLOCK = 22
WEEK = 7 * DAY
N_ROUTES = 1000
ROWS_PER_WINDOW = N_ROUTES * H

DRIVE_MOUNT = Path("/content/drive")


def is_project(path):
    return (path / "data").is_dir() and (path / "solutions").is_dir()


def resolve_project():
    cwd = Path.cwd().resolve()
    requested = None if PROJECT_DIR is None else Path(PROJECT_DIR).expanduser()
    if requested is None:
        for folder in (cwd, *cwd.parents):
            if is_project(folder):
                return folder, folder / "solutions"
    elif is_project(requested):
        folder = requested.resolve()
        return folder, folder / "solutions"

    try:
        from google.colab import drive
    except ImportError:
        if requested is not None:
            raise FileNotFoundError(f"В {requested} должны быть папки data и solutions.")
        folder = cwd.parent if cwd.name == "solutions" else cwd
        return folder, folder / "solutions"

    drive_root = DRIVE_MOUNT / "MyDrive"
    if not drive_root.is_dir():
        drive.mount(str(DRIVE_MOUNT))

    if requested is not None:
        folder = requested if requested.is_absolute() else drive_root / requested
        if not is_project(folder):
            raise FileNotFoundError(f"В {folder} должны быть папки data и solutions.")
        return folder, folder / "solutions"

    candidates = [folder for folder in (drive_root, *drive_root.iterdir()) if is_project(folder)]
    named = [folder for folder in candidates
             if (folder / "solutions/classic_ml_v8.ipynb").is_file()]
    matches = named or candidates
    if len(matches) == 1:
        return matches[0], matches[0] / "solutions"
    raise FileNotFoundError("Задайте PROJECT_DIR путём к папке с data и solutions на Google Диске.")


ROOT, SOLUTIONS = resolve_project()
DATA_DIR = Path(os.environ.get("DATA_DIR", ROOT / "data")).expanduser().resolve()
SUBMISSION = SOLUTIONS / "classic_ml_v8.csv"

## Данные

Папки `data/` и `solutions/` лежат рядом. В Colab при необходимости задайте `PROJECT_DIR`.

In [ ]:
_assets_directory = TemporaryDirectory(prefix="rwb-classic-v8-")
ASSETS = Path(_assets_directory.name)
with ZipFile(BytesIO(b64decode(
    b"..."
))) as archive:
    archive.extractall(ASSETS)

SOURCE = ASSETS / "residual_v8"
REFERENCE_TEST = ASSETS / "data/test_solo_track.parquet"
TEST_PATH = next(
    (path for path in (DATA_DIR / "test_solo_track.parquet", DATA_DIR / "test_solo_track(1).parquet")
     if path.is_file()),
    REFERENCE_TEST,
)

## Признаки и модель

In [ ]:
def stamp(o):
    return pd.Timestamp(times[0]) + o * step


def features(o):
    assert o >= 48
    rr = np.repeat(np.arange(R), H)
    hh = np.tile(np.arange(H), R)
    tt = pd.date_range(stamp(o), periods=H, freq=step)
    x = {'tod': np.tile(tt.hour * 2 + tt.minute // 30, R), 'dow': np.tile(tt.dayofweek, R), 'horizon': hh + 1}
    for j, s in enumerate(S, 1):
        x[f'status_{j}_ratio'] = np.repeat(s[:, o - 1] / (s[:, o - 6:o].mean(1) + 1), H)
    for lag in [1, 2, 4, 8, 16, 48]:
        x[f'target_origin_lag_{lag}'] = np.repeat(Y[:, o - lag], H)
    for w in [4, 12, 24, 48]:
        x[f'target_history_ma_{w}'] = np.repeat(Y[:, o - w:o].mean(1), H)
    assert H <= 48
    x['target_same_time_yesterday'] = Y[rr, o + hh - 48]
    x['route_id'] = np.repeat(routes, H)
    X = pd.DataFrame(x)
    return X


def causal_flow(y):
    n_routes, n_times = y.shape
    q = np.empty(y.shape, dtype=np.float32)
    idx = np.arange(n_times)
    sign = np.where(idx % 2 == 0, 1.0, -1.0)
    for r in range(n_routes):
        b = lfilter([1], [1, 1], y[r].astype(float))
        lo = np.maximum.accumulate(np.where(idx % 2 == 0, -b, -np.inf))
        hi = np.minimum.accumulate(np.where(idx % 2 == 1, b, np.inf))
        mid = (lo + hi) / 2
        mid[~np.isfinite(mid)] = 0
        q[r] = np.maximum(b + sign * mid, 0)
    return q


def expanded(o):
    x = features(o)
    rr = np.repeat(np.arange(R), H)
    hh = np.tile(np.arange(H), R)
    for j, s in enumerate(S[:3], 1):
        for lag in [1, 2, 4, 8]:
            x[f's{j}_lag{lag}'] = np.repeat(s[:, o - lag], H)
        for w in [4, 12, 48]:
            x[f's{j}_mean{w}'] = np.repeat(s[:, o - w:o].mean(1), H)
    for lag in [3, 6, 12, 24, 96, 336]:
        x[f'y_lag{lag}'] = np.repeat(Y[:, o - lag], H)
    for w in [96, 336]:
        x[f'y_mean{w}'] = np.repeat(Y[:, o - w:o].mean(1), H)
    for lag in [1, 2, 4]:
        x[f'q_lag{lag}'] = np.repeat(Q[:, o - lag], H)
    x['y_week'] = Y[rr, o + hh - 336]
    for col in x.columns:
        if col != 'route_id':
            x[col] = x[col].astype(np.float32)
    return x


def make(os):
    return (pd.concat([expanded(int(o)) for o in os], ignore_index=True), np.concatenate([Y[:, o:o + H].ravel() for o in os]))


def design(y, s, ix, ylags=24, slags=8):
    cols = [y[ix - l] for l in range(1, ylags + 1)]
    cols.extend((y[ix - l] for l in [48, 49, 96, 336]))
    for sj in s:
        cols.extend((sj[ix - l] for l in range(1, slags + 1)))
    for period, ks in [(48, 3), (336, 1)]:
        for k in range(1, ks + 1):
            cols.extend([np.sin(2 * np.pi * ix * k / period), np.cos(2 * np.pi * ix * k / period)])
    return np.column_stack(cols)


def ridge(x, y, z, alpha):
    mu = x.mean(0)
    sd = x.std(0)
    sd[sd < 1e-08] = 1
    x = np.column_stack([(x - mu) / sd, np.ones(len(x))])
    z = np.column_stack([(z - mu) / sd, np.ones(len(z))])
    reg = np.eye(x.shape[1]) * alpha
    reg[-1, -1] = 1e-08
    co = solve(x.T @ x + reg, x.T @ y, assume_a='pos')
    return np.maximum(z @ co, 0)


def local_predict(cutoff, val_origins):
    idx = np.arange(max(336, cutoff - 56 * 48), cutoff - H + 1)
    ix = np.r_[idx, val_origins]
    pred = np.empty((len(val_origins), R, H))
    for r in range(R):
        x = design(Y[r], [s[r] for s in S[:3]], ix)
        y = np.stack([Y[r, idx + h] for h in range(H)], axis=1)
        pred[:, r, :] = ridge(x[:len(idx)], y, x[len(idx):], 30)
    return pred.ravel()


def load_data():
    global R, T, Y, S, Q, routes, times, step, origins
    train_path = DATA_DIR / "train_solo_track.parquet"
    if not train_path.is_file():
        raise FileNotFoundError(f"Для переобучения положите обучающий Parquet в {train_path}.")
    tr = pd.read_parquet(train_path).sort_values(['route_id', 'timestamp'])
    te = pd.read_parquet(TEST_PATH)
    routes = np.sort(tr.route_id.unique())
    times = np.sort(tr.timestamp.unique())
    R = len(routes)
    T = len(times)
    assert not tr.duplicated(['route_id', 'timestamp']).any()
    assert all((np.array_equal(g.timestamp.values, times) for _, g in tr.groupby('route_id')))
    step = pd.Timedelta(times[1] - times[0])
    assert np.all(np.diff(times) == np.timedelta64(step.value, 'ns'))
    assert te.id.is_unique and (not te.duplicated(['route_id', 'timestamp']).any())
    future = np.sort(te.timestamp.unique())
    assert len(future) == H
    assert set(te.route_id) == set(routes) and (te.groupby('route_id').size() == H).all()
    assert all((np.array_equal(np.sort(g.timestamp.values), future) for _, g in te.groupby('route_id')))
    assert pd.Timestamp(future[0]) == pd.Timestamp(times[-1]) + step
    assert np.all(np.diff(future) == np.timedelta64(step.value, 'ns'))
    Y = tr.target_1h.to_numpy().reshape(R, T)
    S = [tr[f'status_{i}'].to_numpy().reshape(R, T) for i in range(1, 7)]
    del tr
    gc.collect()
    Q = causal_flow(Y)
    start = pd.Timestamp(future[0])
    origins = np.array([o for o in range(48, T - H + 1) if stamp(o).time() == start.time()])
    validation = np.array([o for o in origins if stamp(o).weekday() == start.weekday()])[-4:]
    assert len(validation) == 4

In [ ]:
def read_targets(path):
    with np.load(path) as data:
        return {k: data[k] for k in data.files}


def residual_features(origin, global_pred, ridge_pred, cutoff):
    x = expanded(origin)
    last_week = Y[:, origin - WEEK:origin]
    scale = np.repeat(np.maximum(last_week.mean(axis=1), 1), H)
    x["ridge_pred"] = ridge_pred
    x["global_pred"] = global_pred
    x["disagreement"] = global_pred - ridge_pred
    x["route_scale"] = scale
    x["route_logscale"] = np.log1p(scale)
    x["ridge_relative"] = ridge_pred / scale
    x["global_relative"] = global_pred / scale
    x["recent_cv"] = np.repeat(last_week.std(axis=1), H) / scale
    x["zero_fraction"] = np.repeat((last_week == 0).mean(axis=1), H)
    x["mean_shift"] = np.repeat(Y[:, origin - DAY:origin].mean(axis=1), H) / scale
    x["model_age_days"] = (origin - cutoff) / DAY
    x["local_history_days"] = min(56, (cutoff - WEEK) / DAY)
    for name in x:
        if name != "route_id":
            x[name] = x[name].astype(np.float32)
    return x, scale


def weekly_predictions(cutoff, predict_origins, threads):
    fit_origins = origins[(origins >= WEEK) & (origins + H <= cutoff)]
    assert len(fit_origins) and cutoff <= predict_origins.min()
    assert np.all(fit_origins + H <= cutoff)
    features, actual = make(fit_origins)
    model = lgb.train(
        dict(objective="regression", num_leaves=31, min_data_in_leaf=300,
             lambda_l2=20, learning_rate=.04, feature_fraction=.9,
             verbosity=-1, num_threads=threads, seed=42),
        lgb.Dataset(features, label=actual, categorical_feature=["route_id"]),
        num_boost_round=450,
    )
    del features, actual
    global_pred = np.stack([
        np.maximum(model.predict(expanded(int(o)), num_threads=threads), 0)
        for o in predict_origins
    ])
    del model
    gc.collect()
    ridge = local_predict(cutoff, predict_origins).reshape(len(predict_origins), R * H)
    return global_pred, ridge


def prepare_history():
    layout = read_targets(ASSETS / "reference/train_layout.npz")
    original_origins = layout["origins"]
    assert int(layout["T"]) == T and H == 8 and R == N_ROUTES
    assert np.all(original_origins % DAY == CLOCK)
    assert np.all(np.diff(original_origins) == DAY)
    assert stamp(0).hour == 0 and stamp(0).minute == 0

    frames, actual, ridge, global_prediction, scales = [], [], [], [], []
    for cutoff in original_origins[::7]:
        selected = original_origins[(original_origins >= cutoff) & (original_origins < cutoff + WEEK)]
        print(f"{stamp(cutoff).date()} · {len(selected)} отсечек", flush=True)
        global_block, ridge_block = weekly_predictions(cutoff, selected, BASE_THREADS)
        for origin, gp, rp in zip(selected, global_block, ridge_block):
            frame, scale = residual_features(int(origin), gp, rp, cutoff)
            frames.append(frame)
            actual.append(Y[:, origin:origin + H].ravel())
            ridge.append(rp)
            global_prediction.append(gp)
            scales.append(scale)
    features = pd.concat(frames, ignore_index=True)
    targets = {
        "origin": np.repeat(original_origins, ROWS_PER_WINDOW),
        "y": np.concatenate(actual),
        "ridge": np.concatenate(ridge),
        "global_prediction": np.concatenate(global_prediction),
        "scale": np.concatenate(scales),
        "folds": layout["folds"],
        "T": layout["T"],
    }
    return features, targets, layout["totals"]

In [ ]:
def transform(features, scale):
    features = features.copy()
    columns = [
        c for c in features
        if c.startswith(("target_", "y_", "q_"))
        or (len(c) > 2 and c[0] == "s" and c[1].isdigit() and c[2] == "_")
        or c in ("ridge_pred", "global_pred", "disagreement")
    ]
    features[columns] = features[columns].div(scale, axis=0).astype(np.float32)
    return features


def fit_residual(features, history, cutoff, validation, validation_scale, recipe):
    mask = history["origin"] + H <= cutoff
    scale = history["scale"][mask]
    train = transform(features.loc[mask], scale)
    validation = transform(validation, validation_scale)
    target = (history["y"][mask] - history["ridge"][mask]) / scale
    weights = scale / scale.mean()
    params = dict(
        objective="regression_l1", learning_rate=.035, num_leaves=31,
        min_data_in_leaf=250, lambda_l2=20, feature_fraction=.9,
        verbosity=-1, num_threads=THREADS, seed=2026,
    )
    print(f"{mask.sum():,} строк · {recipe['trees']} деревьев", flush=True)
    model = lgb.train(
        params,
        lgb.Dataset(train, label=target, weight=weights, categorical_feature=["route_id"]),
        num_boost_round=recipe["trees"],
    )
    return model.predict(validation, num_iteration=recipe["trees"], num_threads=THREADS) * validation_scale


def score(y,p):return (abs(y-p).sum((1,2))+abs(y.sum((1,2))-p.sum((1,2))))/y.sum((1,2))


def anchor(p,total):
 p=np.maximum(p,0);return p*(total/p.sum((1,2)))[:,None,None]


def make_prediction(r,total,baseline,delta,recipe):
 corrected=anchor(r+recipe['alpha']*delta,total)
 return (1-recipe['weight'])*baseline+recipe['weight']*corrected

## Прогноз и выгрузка

По умолчанию используется сохранённая модель. Для переобучения добавьте `train_solo_track.parquet` в `data/` и установите `RETRAIN = True`. Для повторной проверки выбранных настроек также установите `RECHECK = True`.

In [ ]:
def compare_folds(features, history, totals, recipe):
    weights = np.array([.5] * 4 + [.25] * 4)[None, None, :]
    v7_scores, v8_scores = [], []
    for cutoff, total in zip(history["folds"], totals):
        origin = history["origin"]
        mask = (origin >= cutoff) & (origin < cutoff + WEEK)
        assert mask.sum() == 7 * ROWS_PER_WINDOW and np.all(origin[mask] % DAY == CLOCK)
        actual, ridge, global_prediction = (
            history[key][mask].reshape(-1, N_ROUTES, H)
            for key in ("y", "ridge", "global_prediction")
        )
        baseline = anchor((1 - weights) * global_prediction + weights * ridge, total)
        delta = fit_residual(features, history, int(cutoff), features.loc[mask], history["scale"][mask], recipe)
        prediction = make_prediction(ridge, total, baseline, delta.reshape(actual.shape), recipe)
        v7_scores.extend(score(actual, baseline))
        v8_scores.extend(score(actual, prediction))
    return pd.DataFrame(
        {"Подбор": [np.mean(v7_scores[:14]), np.mean(v8_scores[:14])],
         "Контроль": [np.mean(v7_scores[14:]), np.mean(v8_scores[14:])]},
        index=["v7", "v8"],
    )


def write_submission(test, delta, test_data, recipe):
    v7 = pd.read_csv(ASSETS / "horizon_v7/submission_horizon_v7.csv")
    ordered = test.merge(v7, on="id", validate="one_to_one").sort_values(["route_id", "timestamp"])
    baseline = ordered.y_pred.to_numpy().reshape(1, N_ROUTES, H)
    prediction = make_prediction(
        test_data["ridge"].reshape(1, N_ROUTES, H),
        baseline.sum((1, 2)),
        baseline,
        delta.reshape(1, N_ROUTES, H),
        recipe,
    )
    lookup = ordered[["id"]].assign(y_pred=prediction.ravel())
    result = test[["id"]].merge(lookup, on="id", validate="one_to_one")
    assert len(result) == ROWS_PER_WINDOW and result.id.is_unique and result.id.equals(test.id)
    assert np.isfinite(result.y_pred).all() and result.y_pred.ge(0).all()
    assert np.isclose(result.y_pred.sum(), baseline.sum(), rtol=1e-12)
    SOLUTIONS.mkdir(parents=True, exist_ok=True)
    temporary = SUBMISSION.with_suffix(".csv.tmp")
    result.to_csv(temporary, index=False)
    temporary.replace(SUBMISSION)
    print(f"{SUBMISSION} · {len(result):,} строк")

In [ ]:
test = pd.read_parquet(TEST_PATH)
if TEST_PATH != REFERENCE_TEST:
    columns = ["id", "route_id", "timestamp"]
    expected = pd.read_parquet(REFERENCE_TEST, columns=columns).sort_values("id").reset_index(drop=True)
    supplied = test[columns].sort_values("id").reset_index(drop=True)
    pd.testing.assert_frame_equal(supplied, expected, check_dtype=False)
test_data = read_targets(SOURCE / "test_components.npz")
test_features = pd.read_parquet(SOURCE / "test_features.parquet")
report = json.loads((SOURCE / "report.json").read_text())
assert report["pass_gate"]
recipe = report["selected"]["recipe"]
assert recipe["mode"] == "norm"

if RETRAIN:
    load_data()
    features, targets, totals = prepare_history()
    assert features.columns.tolist() == test_features.columns.tolist()
    if RECHECK:
        print(compare_folds(features, targets, totals, recipe).to_string())
    delta = fit_residual(features, targets, int(targets["T"]), test_features, test_data["scale"], recipe)
else:
    model = lgb.Booster(model_file=str(SOURCE / "residual_model_None.txt"))
    assert model.feature_name() == test_features.columns.tolist()
    matrix = transform(test_features, test_data["scale"])
    delta = model.predict(matrix, num_iteration=recipe["trees"], num_threads=THREADS) * test_data["scale"]

print(f"Сохранённая контрольная метрика v8: {report['check']['new_mean']:.6f}")
write_submission(test, delta, test_data, recipe)